# Model YAML Analysis

Analyzes all YAML files under `providers/` to extract unique param keys, types, fields, option structures, and features for tightening CUE validation.

In [13]:
import yaml
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

PROVIDERS_DIR = Path("../providers")

configs = []
for f in PROVIDERS_DIR.rglob("*.yaml"):
    with open(f) as fh:
        data = yaml.safe_load(fh)
        if data:
            configs.append({"path": str(f), "data": data})

print(f"Parsed {len(configs)} YAML files")

Parsed 2227 YAML files


## 1. Param Keys

In [14]:
param_keys = Counter()
param_key_examples = defaultdict(list)

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "key" in p:
            k = p["key"]
            param_keys[k] += 1
            if len(param_key_examples[k]) < 1:
                param_key_examples[k].append(entry["path"])

print(f"Unique param keys: {len(param_keys)}\n")
for key, count in param_keys.most_common():
    print(f'  "{key}"  ({count})  e.g. {param_key_examples[key][0]}')

Unique param keys: 27

  "max_tokens"  (445)  e.g. ../providers/cohere/command-a-reasoning-08-2025.yaml
  "response_format"  (80)  e.g. ../providers/openrouter/default.yaml
  "max_completion_tokens"  (42)  e.g. ../providers/azure-open-ai/o4-mini.yaml
  "temperature"  (38)  e.g. ../providers/cohere/default.yaml
  "thinking"  (37)  e.g. ../providers/google-vertex/gemini-2.5-flash.yaml
  "tool_choice"  (22)  e.g. ../providers/groq/llama-3.3-70b-versatile.yaml
  "top_p"  (21)  e.g. ../providers/openrouter/default.yaml
  "frequency_penalty"  (18)  e.g. ../providers/openrouter/default.yaml
  "presence_penalty"  (18)  e.g. ../providers/openrouter/default.yaml
  "reasoning_effort"  (17)  e.g. ../providers/openrouter/openai/gpt-5.3-codex.yaml
  "stream"  (16)  e.g. ../providers/cohere/default.yaml
  "stop"  (14)  e.g. ../providers/openrouter/default.yaml
  "n"  (13)  e.g. ../providers/openrouter/default.yaml
  "reasoning"  (12)  e.g. ../providers/openrouter/moonshotai/kimi-k2-thinking.yaml
  "v

## 2. Param Types

In [15]:
param_types = Counter()
param_type_examples = defaultdict(list)

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "type" in p:
            t = p["type"]
            label = t if isinstance(t, str) else f"(object: {sorted(t.keys())})"
            param_types[label] += 1
            if len(param_type_examples[label]) < 1:
                param_type_examples[label].append(f'{entry["path"]} (key={p.get("key")})')

print(f"Unique param type values: {len(param_types)}\n")
for t, count in param_types.most_common():
    print(f'  {t:40s}  ({count})  e.g. {param_type_examples[t][0]}')

Unique param type values: 7

  string                                    (95)  e.g. ../providers/openrouter/default.yaml (key=response_format)
  boolean                                   (21)  e.g. ../providers/cohere/default.yaml (key=stream)
  non-view-manage-data                      (15)  e.g. ../providers/groq/llama-3.3-70b-versatile.yaml (key=tool_choice)
  array-of-strings                          (14)  e.g. ../providers/openrouter/default.yaml (key=stop)
  number                                    (12)  e.g. ../providers/aws-bedrock/us.anthropic.claude-opus-4-5-20251101-v1:0.yaml (key=thinking)
  json                                      (7)  e.g. ../providers/aws-bedrock/us.anthropic.claude-3-haiku-20240307-v1:0.yaml (key=tool_choice)
  object                                    (1)  e.g. ../providers/openrouter/moonshotai/kimi-k2-thinking.yaml (key=reasoning)


## 3. Param Object Fields

In [16]:
param_fields = Counter()
param_field_examples = defaultdict(list)

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict):
            for field in p.keys():
                param_fields[field] += 1
                if len(param_field_examples[field]) < 1:
                    param_field_examples[field].append(entry["path"])

print(f"Unique fields in param objects: {len(param_fields)}\n")
for field, count in param_fields.most_common():
    print(f"  {field:25s}  ({count:4d})  e.g. {param_field_examples[field][0]}")

Unique fields in param objects: 12

  key                        ( 827)  e.g. ../providers/cohere/command-a-reasoning-08-2025.yaml
  maxValue                   ( 611)  e.g. ../providers/cohere/command-a-reasoning-08-2025.yaml
  defaultValue               ( 508)  e.g. ../providers/cohere/command-a-reasoning-08-2025.yaml
  minValue                   ( 315)  e.g. ../providers/cohere/command-a-reasoning-08-2025.yaml
  skipValues                 ( 178)  e.g. ../providers/openrouter/default.yaml
  type                       ( 165)  e.g. ../providers/cohere/default.yaml
  options                    ( 118)  e.g. ../providers/openrouter/default.yaml
  withdrawParams             (  26)  e.g. ../providers/aws-bedrock/us.anthropic.claude-3-7-sonnet-20250219-v1:0.yaml
  properties                 (  25)  e.g. ../providers/google-vertex/gemini-2.5-flash.yaml
  rule                       (  24)  e.g. ../providers/groq/llama-3.3-70b-versatile.yaml
  enum                       (  23)  e.g. ../providers

## 4. defaultValue — types and example values

In [17]:
dv_by_type = defaultdict(set)

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "defaultValue" in p:
            v = p["defaultValue"]
            dv_by_type[type(v).__name__].add(repr(v))

for t in sorted(dv_by_type):
    vals = sorted(dv_by_type[t])
    print(f"  {t}: {vals[:15]}")
    if len(vals) > 15:
        print(f"    ... and {len(vals) - 15} more")

  NoneType: ['None']
  bool: ['False', 'True']
  float: ['0.1', '0.3', '0.35', '0.5', '0.6', '0.7', '0.75', '0.8', '0.9', '0.95']
  int: ['0', '1', '1000', '10000', '100000', '1024', '128', '128000', '16', '16384', '20000', '2048', '250', '256', '40']
    ... and 9 more
  str: ["'high'", "'medium'", "'none'", "'png'"]


## 5. skipValues — item types and example values

In [18]:
sv_by_type = defaultdict(set)

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict):
            for item in (p.get("skipValues") or []):
                sv_by_type[type(item).__name__].add(repr(item))

for t in sorted(sv_by_type):
    print(f"  {t}: {sorted(sv_by_type[t])}")

  NoneType: ['None']
  bool: ['True']
  list: ['[]']
  str: ["'disabled'"]


## 6. enum — item types and example values

In [19]:
enum_by_type = defaultdict(set)
enum_combos = Counter()

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "enum" in p:
            combo = tuple(repr(x) for x in p["enum"])
            enum_combos[combo] += 1
            for item in p["enum"]:
                enum_by_type[type(item).__name__].add(repr(item))

print("Item types:")
for t in sorted(enum_by_type):
    print(f"  {t}: {sorted(enum_by_type[t])}")

print(f"\nUnique enum combinations ({len(enum_combos)}):")
for combo, count in enum_combos.most_common():
    print(f"  {list(combo)}  ({count})")

Item types:
  NoneType: ['None']
  str: ["'disabled'", "'enabled'", "'high'", "'low'", "'medium'"]

Unique enum combinations (2):
  ["'enabled'", "'disabled'"]  (12)
  ["'low'", "'medium'", "'high'", 'None']  (11)


## 7. Options — fields, value types, and schema structures

In [20]:
opt_fields = Counter()
opt_value_types = defaultdict(set)
opt_schema_examples = []

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict):
            for opt in (p.get("options") or []):
                if isinstance(opt, dict):
                    for field in opt.keys():
                        opt_fields[field] += 1
                    v = opt.get("value")
                    opt_value_types[type(v).__name__].add(repr(v))
                    if "schema" in opt and len(opt_schema_examples) < 5:
                        opt_schema_examples.append({
                            "file": entry["path"],
                            "param_key": p.get("key"),
                            "option_name": opt.get("name"),
                            "schema": opt["schema"]
                        })

print("Option fields:")
for field, count in opt_fields.most_common():
    print(f"  {field:15s}  ({count})")

print("\nOption value types:")
for t in sorted(opt_value_types):
    vals = sorted(opt_value_types[t])
    print(f"  {t}: {vals[:10]}")

print("\nExample schema structures:")
for ex in opt_schema_examples:
    print(f"  {ex['file']} (key={ex['param_key']}, option={ex['option_name']})")
    pprint(ex["schema"], indent=4, width=100)
    print()

Option fields:
  name             (365)
  value            (365)
  schema           (164)
  params           (63)
  type             (8)

Option value types:
  NoneType: ['None']
  str: ["'auto'", "'custom'", "'high'", "'json_object'", "'json_schema'", "'low'", "'medium'", "'none'", "'required'", "'xhigh'"]

Example schema structures:
  ../providers/openrouter/default.yaml (key=response_format, option=JSON Object)
{'properties': {'type': {'type': 'string', 'value': 'json_object'}}, 'type': 'object'}

  ../providers/groq/llama-3.3-70b-versatile.yaml (key=tool_choice, option=Custom)
{'type': 'json'}

  ../providers/groq/llama-3.3-70b-versatile.yaml (key=response_format, option=JSON Object)
{'properties': {'type': {'type': 'string', 'value': 'json_object'}}, 'type': 'object'}

  ../providers/groq/llama-3.1-8b-instant.yaml (key=tool_choice, option=Custom)
{'type': 'json'}

  ../providers/groq/llama-3.1-8b-instant.yaml (key=response_format, option=JSON Object)
{'properties': {'type': {'type

## 8. Options — nested params

In [21]:
opt_params_examples = []

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict):
            for opt in (p.get("options") or []):
                if isinstance(opt, dict) and "params" in opt and len(opt_params_examples) < 3:
                    opt_params_examples.append({
                        "file": entry["path"],
                        "param_key": p.get("key"),
                        "option_name": opt.get("name"),
                        "nested_params": opt["params"]
                    })

for ex in opt_params_examples:
    print(f"  {ex['file']} (key={ex['param_key']}, option={ex['option_name']})")
    pprint(ex["nested_params"], indent=4, width=100)
    print()

  ../providers/groq/llama-3.1-8b-instant.yaml (key=response_format, option=JSON Schema)
{'defaultValue': None, 'key': 'json_schema', 'skipValues': [None], 'type': 'json'}

  ../providers/google-vertex/gemini-1.5-flash-002.yaml (key=response_format, option=JSON Schema)
{'defaultValue': None, 'key': 'json_schema', 'skipValues': [None], 'type': 'json'}

  ../providers/google-vertex/gemini-2.0-flash-001.yaml (key=response_format, option=JSON Schema)
{'defaultValue': None, 'key': 'json_schema', 'skipValues': [None], 'type': 'json'}



## 9. properties — structures and example values

In [22]:
prop_subfields = Counter()
shown = 0

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "properties" in p:
            for prop_name, prop_val in p["properties"].items():
                if isinstance(prop_val, dict):
                    for sf in prop_val.keys():
                        prop_subfields[sf] += 1
            if shown < 3:
                print(f"  {entry['path']} (key={p['key']})")
                pprint(p["properties"], indent=4, width=100)
                print()
                shown += 1

print(f"Sub-fields inside properties values:")
for sf, count in prop_subfields.most_common():
    print(f"  {sf:15s}  ({count})")

  ../providers/google-vertex/gemini-2.5-flash.yaml (key=thinking)
{   'budget_tokens': {'maxValue': 24576, 'minValue': 0, 'type': 'number'},
    'type': {'enum': ['enabled', 'disabled'], 'type': 'string'}}

  ../providers/google-vertex/gemini-2.5-pro.yaml (key=thinking)
{   'budget_tokens': {'maxValue': 32768, 'minValue': 128, 'type': 'number'},
    'type': {'enum': ['enabled'], 'type': 'string'}}

  ../providers/google-vertex/gemini-2.5-flash-lite.yaml (key=thinking)
{   'budget_tokens': {'maxValue': 24576, 'minValue': 0, 'type': 'number'},
    'type': {'enum': ['enabled', 'disabled'], 'type': 'string'}}

Sub-fields inside properties values:
  type             (49)
  enum             (25)
  maxValue         (24)
  minValue         (24)


## 10. nestedOptions — structure and example values

In [23]:
nested_opt_fields = Counter()
nested_opt_value_keys = Counter()
shown = 0

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "nestedOptions" in p:
            for no in p["nestedOptions"]:
                if isinstance(no, dict):
                    for field in no.keys():
                        nested_opt_fields[field] += 1
                    if isinstance(no.get("value"), dict):
                        for vk in no["value"].keys():
                            nested_opt_value_keys[vk] += 1
            if shown < 2:
                print(f"  {entry['path']} (key={p['key']})")
                pprint(p["nestedOptions"], indent=4, width=100)
                print()
                shown += 1

print(f"nestedOption item fields: {dict(nested_opt_fields)}")
print(f"nestedOption value keys:  {dict(nested_opt_value_keys)}")

  ../providers/azure-open-ai/gpt-5.yaml (key=reasoning)
[   {'value': {'effort': 'minimal'}, 'view': 'minimal'},
    {'value': {'effort': 'low'}, 'view': 'low'},
    {'value': {'effort': 'medium'}, 'view': 'medium'},
    {'value': {'effort': 'high'}, 'view': 'high'}]

  ../providers/azure-open-ai/gpt-5-chat-latest.yaml (key=reasoning)
[   {'value': {'effort': 'minimal'}, 'view': 'minimal'},
    {'value': {'effort': 'low'}, 'view': 'low'},
    {'value': {'effort': 'medium'}, 'view': 'medium'},
    {'value': {'effort': 'high'}, 'view': 'high'}]

nestedOption item fields: {'value': 44, 'view': 44}
nestedOption value keys:  {'effort': 44}


## 11. rule — structure and values

In [24]:
rule_patterns = Counter()
rule_else_vals = set()
rule_then_vals = set()
rule_cond_vals = set()

for entry in configs:
    for p in (entry["data"].get("params") or []):
        if isinstance(p, dict) and "rule" in p:
            r = p["rule"]
            if r is None:
                rule_patterns["null"] += 1
            elif isinstance(r, dict) and "default" in r:
                d = r["default"]
                rule_patterns["nested {default: {...}}"] += 1
                if isinstance(d, dict):
                    rule_cond_vals.add(repr(d.get("condition")))
                    rule_else_vals.add(repr(d.get("else")))
                    rule_then_vals.add(repr(d.get("then")))
            else:
                rule_patterns[f"other: {repr(r)}"] += 1

print("Rule patterns:")
for pat, count in rule_patterns.most_common():
    print(f"  {pat:35s}  ({count})")

print(f"\nrule.default.condition values: {sorted(rule_cond_vals)}")
print(f"rule.default.else values:      {sorted(rule_else_vals)}")
print(f"rule.default.then values:      {sorted(rule_then_vals)}")

Rule patterns:
  nested {default: {...}}              (24)

rule.default.condition values: ["'tools'"]
rule.default.else values:      ['None', 'True']
rule.default.then values:      ["'auto'", 'True']


## 12. Features

In [25]:
feature_values = Counter()

for entry in configs:
    for f in (entry["data"].get("features") or []):
        if isinstance(f, str):
            feature_values[f] += 1

modality_features = {
    "text", "image_input", "vision", "embedding_image_input",
    "image", "audio_input", "audio_output",
    "pdf_input", "pdf", "doc", "code"
}

print(f"Unique feature values: {len(feature_values)}\n")
print("Capabilities:")
for feat, count in feature_values.most_common():
    if feat not in modality_features:
        print(f'  {feat:30s}  ({count})')

print("\nModality-related (to move to #Modalities):")
for feat, count in feature_values.most_common():
    if feat in modality_features:
        print(f'  {feat:30s}  ({count})')

Unique feature values: 21

Capabilities:
  function_calling                (1181)
  chat                            (1125)
  tool_choice                     (993)
  response_schema                 (700)
  system_messages                 (521)
  tools                           (476)
  prompt_caching                  (340)
  parallel_function_calling       (189)
  assistant_prefill               (148)
  cache_control                   (30)

Modality-related (to move to #Modalities):
  vision                          (673)
  image_input                     (429)
  text                            (387)
  code                            (196)
  image                           (193)
  pdf_input                       (151)
  audio_input                     (97)
  doc                             (72)
  audio_output                    (56)
  pdf                             (54)
  embedding_image_input           (33)


## 13. Summary for CUE schema

In [26]:
print("=" * 60)
print("SUMMARY FOR CUE SCHEMA")
print("=" * 60)

print("\n#ParamKey enum values:")
for key in sorted(param_keys.keys()):
    print(f'  "{key}" |')

print("\n#ParamType enum values:")
for t in sorted(t for t in param_types.keys() if not t.startswith("(")):
    print(f'  "{t}" |')

print("\nAll param object fields:")
for field in sorted(param_fields.keys()):
    print(f"  {field}")

print("\nAll option object fields:")
for field in sorted(opt_fields.keys()):
    print(f"  {field}")

print("\nCapability #Feature values:")
for f in sorted(set(feature_values.keys()) - modality_features):
    print(f'  "{f}" |')

print("\nModality-related features:")
for f in sorted(modality_features & set(feature_values.keys())):
    print(f'  "{f}" ({feature_values[f]})')

SUMMARY FOR CUE SCHEMA

#ParamKey enum values:
  "clear_thinking" |
  "creativity" |
  "disable_reasoning" |
  "frequency_penalty" |
  "grow_mask" |
  "logit_bias" |
  "max_completion_tokens" |
  "max_tokens" |
  "max_tokens_per_doc" |
  "min_tokens" |
  "n" |
  "output_format" |
  "parallel_tool_calls" |
  "presence_penalty" |
  "reasoning" |
  "reasoning_effort" |
  "response_format" |
  "safe_prompt" |
  "seed" |
  "stop" |
  "stream" |
  "temperature" |
  "thinking" |
  "tool_choice" |
  "top_k" |
  "top_p" |
  "verbosity" |

#ParamType enum values:
  "array-of-strings" |
  "boolean" |
  "json" |
  "non-view-manage-data" |
  "number" |
  "object" |
  "string" |

All param object fields:
  defaultValue
  enum
  key
  maxValue
  minValue
  nestedOptions
  options
  properties
  rule
  skipValues
  type
  withdrawParams

All option object fields:
  name
  params
  schema
  type
  value

Capability #Feature values:
  "assistant_prefill" |
  "cache_control" |
  "chat" |
  "function_call